# Fine-tune GPT-2 Vietnamese for Math — V6

**Core idea:** stronger PEFT LoRA answer-only tuning.

V6 intentionally removes teacher-KD and RL. It optimizes the actual scoring
target: stable extraction of the final numeric answer.

- `train.json`
- `valid.json`
- optional `test.json`
- fixed base model `GPT2_vietnamese/`

The notebook still emits the required `model_output` schema with a clear final
anchor `Đáp án là: <number>`.


In [ ]:
# ============================================================
# 0. Install/import dependencies
# ============================================================
import os, sys, json, math, time, re, random, hashlib, inspect, shutil
from pathlib import Path
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

PIP_INSTALL_DEPS = False  # Kaggle official run: Internet OFF; avoid pip overhead
PIP_PACKAGES = ["peft", "accelerate", "datasets", "evaluate"]

if PIP_INSTALL_DEPS:
    import subprocess
    print("[pip] Installing:", " ".join(PIP_PACKAGES))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES])

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
import peft

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print("PEFT :", getattr(peft, "__version__", "unknown"))


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find any path: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/dataset-math",
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "dataset",
)

MODEL_NAME = str(first_existing(
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "GPT2_vietnamese",
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"

# V6: no teacher-KD artifact.
USE_KD = False
REQUIRE_KD_FILE = False

# Run mode: "phase1" validates on valid.json; "phase2" writes test_predictions.json.
RUN_MODE = "phase1"

# Prompt & special tokens
PROMPT_TEMPLATE = "Bài toán: {q}\nLời giải: "
SAFE_EOS_ID = 50256
N_POSITIONS = 1024

# Working dirs / outputs
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
STAGE_A_OUTPUT_DIR = WORKING_DIR / "gpt2_math_lora_v6_stage_a"
SFT_OUTPUT_DIR     = WORKING_DIR / "gpt2_math_lora_v6_answer_only"
FINAL_OUTPUT_DIR   = WORKING_DIR / "gpt2_math_lora_v6_final"

STAGE_A_VALID_OUTPUT_PATH = WORKING_DIR / "valid_output_stage_a.json"
STAGE_A_VALID_REPORT_PATH = WORKING_DIR / "valid_report_stage_a.json"
SFT_VALID_OUTPUT_PATH     = WORKING_DIR / "valid_output_sft.json"
SFT_VALID_REPORT_PATH     = WORKING_DIR / "valid_report_sft.json"
VALID_OUTPUT_PATH         = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH         = WORKING_DIR / "valid_report.json"
VALID_OVERLAP_AUDIT_PATH  = WORKING_DIR / "valid_overlap_audit.json"
RL_LOG_PATH               = WORKING_DIR / "rl_training_log.jsonl"
RL_SUMMARY_PATH           = WORKING_DIR / "rl_reward_summary.json"
TEST_OUTPUT_PATH          = WORKING_DIR / "test_predictions.json"

# Smoke/debug knobs. For a fast local smoke run, set these small.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
DROP_EXACT_DUPLICATES = True
DROP_NON_EXTRACTABLE = True

# Single-stage answer-only LoRA SFT. Increase to 2.5 if Kaggle runtime allows.
STAGE_A_EPOCHS = 2.25  # reduce to 2.0 if final run still exceeds 3h
STAGE_A_LR = 3e-5
MAX_LENGTH_STAGE_A = 256

# V6-simple disables Stage B to remove overhead and spend the budget on one
# answer-only pass. These variables remain only for manifest/backward-compatible
# cells; they are not used for training.
RUN_STAGE_B = False
STAGE_B_TARGET_MODE = "answer_only"
STAGE_B_MAX_RECORDS = None
STAGE_B_EPOCHS = 0.0
STAGE_B_LR = 0.0
MAX_LENGTH_STAGE_B = 256
LOCAL_REASON_MAX_TOKENS = 64
ANSWER_ONLY_REPLAY_RATIO = 0.0

# Trainer
PER_DEVICE_BATCH_SIZE = 16  # fallback: 8 if OOM
GRAD_ACCUM = 2  # fallback: 4 if PER_DEVICE_BATCH_SIZE=8
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn", "c_proj"]

# Stage C: disabled in V6.
RL_ENABLED = False
RL_MAX_PROMPTS = 6000
RL_GROUP_SIZE = 2
RL_PROMPT_BATCH_SIZE = 4
RL_MAX_NEW_TOKENS = 64
RL_LR = 5e-6
RL_MAX_STEPS = 0
RL_TEMPERATURE = 0.8
RL_TOP_P = 0.9
RL_SFT_REPLAY_COEF = 0.20
RL_GRAD_CLIP = 1.0

# Generation defaults for evaluation/submission
MAX_NEW_TOKENS = 32
NUM_BEAMS = 2
DECODE_BATCH_SIZE = 8
NO_REPEAT_NGRAM = 4
REPETITION_PENALTY = 1.15
LENGTH_PENALTY = 0.9
USE_TYPE_AWARE_FEWSHOT = False
USE_SELF_CONSISTENCY = False
SANITIZE_TO_ANSWER_ONLY = True
RUN_STAGE_A_GENERATION_EVAL = False  # avoid duplicate valid generation
RUN_SFT_GENERATION_EVAL = False  # final valid generation still runs once
INFER_FP16 = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# Speed knobs. Harmless on GPUs that do not support TF32.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("TRAIN_FILE       :", TRAIN_FILE)
print("VALID_FILE       :", VALID_FILE)
print("TEST_FILE        :", TEST_FILE, "| exists:", TEST_FILE.exists())
print("MODEL_NAME       :", MODEL_NAME)
print("RUN_MODE         :", RUN_MODE)
print("USE_KD           :", USE_KD)
print("FINAL_OUTPUT_DIR :", FINAL_OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Data loading + robust numeric evaluator
# ============================================================
def load_records(path: str | Path) -> list[dict]:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

ANSWER_ANCHORS = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}
EOS_ARTIFACT_RE = re.compile(r"(?:\s*(?:hue|<\|endoftext\|>|</s>|<pad>))+\s*$", re.IGNORECASE)

def clean_decoded_artifacts(text: str | None) -> str:
    """Remove decoded pseudo-EOS artifacts before answer parsing/saving.

    The task asks us to use SAFE_EOS_ID=50256 because the GPT-2 Vietnamese model
    embedding matrix is sized for ids 0..50256. In this tokenizer, however,
    id 50256 decodes to the ordinary string "hue", not to a special token.
    If generation stops on this id, Hugging Face includes it in decoded text.
    Official scoring needs a clean numeric answer, so strip only trailing
    terminator artifacts.
    """
    text = str(text or "").strip()
    for _ in range(4):
        new_text = EOS_ARTIFACT_RE.sub("", text).strip()
        if new_text == text:
            break
        text = new_text
    return text

def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # V3-stable behavior: id 50256 is required as model EOS/PAD but decodes to
    # the ordinary token "hue" in this tokenizer, so strip it by id before
    # decoding rather than relying on skip_special_tokens.
    return clean_decoded_artifacts(tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True))

def _clean_tail(text: str) -> str:
    text = clean_decoded_artifacts(text).split("\n", 1)[0].strip()
    text = re.sub(r"[.,;:。、“”\"')\]]+$", "", text).strip()
    text = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", text, flags=re.IGNORECASE)
    return text.strip()

def extract_answer(text: str | None) -> str | None:
    if not text:
        return None
    best_end = -1
    best_tail = None
    for pat in ANSWER_ANCHORS:
        for m in pat.finditer(text):
            if m.end() > best_end:
                best_end = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    boxes = BOXED_RE.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(text: str | None) -> float | None:
    if text is None:
        return None
    value = str(text).strip()
    if not value:
        return None
    if re.fullmatch(r"-?\d+,\d+", value):
        try:
            return float(value.replace(",", "."))
        except ValueError:
            return None
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", value):
        try:
            parsed = float(value)
            return parsed if math.isfinite(parsed) else None
        except ValueError:
            return None
    assignment = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", value)
    if assignment:
        value = assignment.group(1).strip()
    if value.startswith("(") and value.endswith(")") and re.search(r"\d\s*,\s*\d", value):
        return None
    if value.startswith("[") and value.endswith("]"):
        return None
    for _ in range(3):
        new_value = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", value)
        if new_value == value:
            break
        value = new_value
    value = re.sub(r"\\text\{[^}]*\}", "", value)
    value = re.sub(r"\\mathrm\{[^}]*\}", "", value)
    value = value.replace("$", "")
    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        value = value.replace(token, "")
    for token in ("\\cdot", "\\times"):
        value = value.replace(token, "*")
    value = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", value)
    value = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", value)
    value = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", value)
    value = value.replace("\\pi", "pi")
    value = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", value)
    value = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", value)
    value = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", value)
    has_period = "." in value
    comma_count = value.count(",")
    if comma_count == 1 and not has_period and re.search(r"\d,\d", value):
        value = re.sub(r"(?<=\d),(?=\d)", ".", value)
    elif comma_count >= 1:
        value = re.sub(r"(?<=\d),(?=\d{3}\b)", "", value)
    value = re.sub(r"\s+", "", value)
    if not value or "," in value:
        return None
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", value)
    if leftover:
        return None
    try:
        parsed = eval(value.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None
    if isinstance(parsed, bool):
        return None
    if isinstance(parsed, (int, float)):
        parsed = float(parsed)
        return parsed if math.isfinite(parsed) else None
    return None

def extract_gold(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("response_vi"))
    return answer, parse_number(answer)

def extract_pred(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("model_output"))
    return answer, parse_number(answer)

def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(error_value: float | None, extractable: bool) -> int:
    if not extractable or error_value is None:
        return 0
    if error_value <= 0.01:
        return 10
    if error_value <= 0.10:
        return 5
    if error_value <= 0.50:
        return 1
    return 0

def evaluate_predictions(pred_items: list[dict], gold_items: list[dict]) -> dict:
    if len(pred_items) != len(gold_items):
        raise ValueError(f"Prediction count {len(pred_items)} != gold count {len(gold_items)}")
    rows, total, extractable, numeric_pairs, rel_errors = [], 0, 0, 0, []
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    by_type = defaultdict(lambda: {"n": 0, "raw_score": 0, "extractable": 0, "bucket_10": 0, "bucket_5": 0, "bucket_1": 0, "bucket_0": 0})
    for pred, gold in zip(pred_items, gold_items):
        gold_answer, gold_num = extract_gold(gold)
        pred_answer, pred_num = extract_pred(pred)
        is_extractable = pred_answer is not None
        error_value = rel_error(pred_num, gold_num)
        score = score_one(error_value, is_extractable)
        t = gold.get("type") or pred.get("type") or "unknown"
        total += score
        extractable += int(is_extractable)
        buckets[score] = buckets.get(score, 0) + 1
        if gold_num is not None and pred_num is not None and error_value is not None:
            numeric_pairs += 1
            rel_errors.append(error_value)
        by_type[t]["n"] += 1
        by_type[t]["raw_score"] += score
        by_type[t]["extractable"] += int(is_extractable)
        by_type[t][f"bucket_{score}"] += 1
        rows.append({
            "id": gold.get("id", pred.get("id")),
            "type": t,
            "gold_answer": gold_answer,
            "gold_num": gold_num,
            "pred_answer": pred_answer,
            "pred_num": pred_num,
            "rel_error": error_value,
            "extractable": is_extractable,
            "score": score,
        })
    n = len(rows)
    by_type_final = {}
    for t, d in sorted(by_type.items()):
        d = dict(d)
        d["score_10"] = d["raw_score"] / d["n"] if d["n"] else 0.0
        by_type_final[t] = d
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": 10 * n,
            "score_10": total / n if n else 0.0,
            "score_pct": total / (10 * n) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "by_type": by_type_final,
        "rows": rows,
    }

def save_eval_report(pred_path: Path, gold_records: list[dict], report_path: Path) -> dict:
    pred_items = json.loads(Path(pred_path).read_text(encoding="utf-8"))
    report = evaluate_predictions(pred_items, gold_records)
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

def canonicalize_answer(gold_num: float | None) -> str | None:
    if gold_num is None:
        return None
    if abs(gold_num - round(gold_num)) < 1e-9:
        return str(int(round(gold_num)))
    return f"{gold_num:g}"

def sanitize_model_output(text: str | None) -> str:
    """Save a clean answer line if the model produced a parseable answer.

    This is prediction post-processing only: it uses the model's own decoded
    answer, never the gold answer. It prevents harmless decoded terminators or
    extra continuation text from making an otherwise numeric prediction
    unparseable by the official-style scorer.
    """
    cleaned = clean_decoded_artifacts(text)
    pred_answer = extract_answer(cleaned)
    pred_num = parse_number(pred_answer)
    canonical = canonicalize_answer(pred_num)
    if canonical is not None:
        return f"Đáp án là: {canonical}"
    return cleaned

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
test_records_for_info = load_records(TEST_FILE) if TEST_FILE.exists() else []

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records), "| test:", len(test_records_for_info))
print("train type distribution:", dict(Counter(r.get("type") for r in train_records).most_common()))


In [ ]:
# ============================================================
# 3. Clean data + build Stage A/B targets
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

def stable_fraction(source_id: int) -> float:
    h = hashlib.sha256(str(source_id).encode()).hexdigest()
    return int(h[:8], 16) / 0x100000000

def build_answer_only_target(canonical_answer: str) -> str:
    return f"Đáp án là: {canonical_answer}"

def save_valid_overlap_audit(train_recs: list[dict], valid_recs: list[dict], path: Path):
    train_q = Counter((r.get("query_vi") or "").strip() for r in train_recs)
    seen = []
    by_type = defaultdict(lambda: {"n": 0, "seen": 0})
    for i, rec in enumerate(valid_recs):
        q = (rec.get("query_vi") or "").strip()
        t = rec.get("type") or "unknown"
        by_type[t]["n"] += 1
        if q in train_q:
            seen.append(i)
            by_type[t]["seen"] += 1
    report = {
        "train_n": len(train_recs),
        "valid_n": len(valid_recs),
        "train_unique_q": len(train_q),
        "valid_seen_query": len(seen),
        "valid_seen_query_pct": len(seen) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_ids_first20": seen[:20],
        "valid_by_type": dict(sorted((k, dict(v)) for k, v in by_type.items())),
    }
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[overlap]", json.dumps({k: report[k] for k in ["train_n", "valid_n", "train_unique_q", "valid_seen_query", "valid_seen_query_pct"]}, ensure_ascii=False))
    return report

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen_exact = set()
    out = []
    dropped_dup = dropped_no_answer = dropped_non_numeric = 0
    for source_id, rec in enumerate(records):
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_no_answer += 1
            continue
        if split == "train" and DROP_EXACT_DUPLICATES:
            key = (q, r)
            if key in seen_exact:
                dropped_dup += 1
                continue
            seen_exact.add(key)
        gold_str, gold_num = extract_gold(rec)
        canonical = canonicalize_answer(gold_num)
        if DROP_NON_EXTRACTABLE and canonical is None:
            dropped_non_numeric += 1
            continue
        out.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
            "_source_id": source_id,
            "_gold_str": gold_str,
            "_gold_num": gold_num,
            "_canonical_answer": canonical,
        })
    print(f"[clean:{split}] kept={len(out)} dropped_dup={dropped_dup} dropped_empty={dropped_no_answer} dropped_non_numeric={dropped_non_numeric}")
    return out

ANSWER_TAIL_RE = re.compile(
    r"(đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?|đ[áa]p\s*[áa]n\s*[:：]|c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?|the\s*answer\s*is\s*[:：]?|####)",
    re.IGNORECASE,
)

def strip_existing_answer_tail(text: str) -> str:
    matches = list(ANSWER_TAIL_RE.finditer(text or ""))
    if matches:
        text = text[:matches[-1].start()]
    text = re.sub(r"\\boxed\{([^{}]+)\}", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_reason_sentences(text: str) -> list[str]:
    raw = re.split(r"(?<=[.!?。])\s+|\n+", text)
    return [s.strip(" -•\t") for s in raw if s.strip(" -•\t")]

def cap_text_tokens(text: str, max_tokens: int) -> str:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text.strip()
    return tokenizer.decode(ids[-max_tokens:], skip_special_tokens=True).strip()

def build_local_compact_reasoning(response: str) -> str:
    body = strip_existing_answer_tail(response)
    if not body:
        return ""
    sentences = split_reason_sentences(body)
    if not sentences:
        return cap_text_tokens(body, LOCAL_REASON_MAX_TOKENS)
    tail = sentences[-6:]
    equationish = [s for s in tail if re.search(r"\d", s) and re.search(r"[+\-*/=×÷^]|\\frac|\\sqrt", s)]
    numeric = [s for s in tail if re.search(r"\d", s)]
    chosen = (equationish[-2:] if equationish else numeric[-2:] if numeric else tail[-2:])
    compact = " ".join(chosen)
    compact = ANSWER_TAIL_RE.sub("", compact).strip()
    return cap_text_tokens(compact, LOCAL_REASON_MAX_TOKENS)

def build_balanced_subset(records: list[dict], max_records: int | None, seed: int) -> list[dict]:
    if max_records is None or max_records >= len(records):
        return list(records)
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for rec in records:
        buckets[rec.get("type") or "unknown"].append(rec)
    for vals in buckets.values():
        rng.shuffle(vals)
    active_types = sorted(buckets)
    out = []
    cursor = 0
    while len(out) < max_records and active_types:
        t = active_types[cursor % len(active_types)]
        if buckets[t]:
            out.append(buckets[t].pop())
        if not buckets[t]:
            active_types.remove(t)
            cursor = 0
        else:
            cursor += 1
    rng.shuffle(out)
    return out

def build_stage_a_records(records: list[dict]) -> list[dict]:
    out = []
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is not None:
            out.append({**rec, "response_vi": build_answer_only_target(canonical), "_stage": "stage_a_answer_only"})
    print(f"[build:stage_a] {len(out)}")
    return out

def build_stage_b_records(records: list[dict]) -> list[dict]:
    selected = build_balanced_subset(records, STAGE_B_MAX_RECORDS, SEED + 5)
    out, reason_used, fallback, replay, answer_only = [], 0, 0, 0, 0
    for rec in selected:
        canonical = rec.get("_canonical_answer")
        if canonical is None:
            continue
        if STAGE_B_TARGET_MODE == "answer_only":
            target = build_answer_only_target(canonical)
            answer_only += 1
        else:
            reason = build_local_compact_reasoning(rec.get("response_vi", ""))
            if reason:
                target = f"{reason}\nĐáp án là: {canonical}"
                reason_used += 1
            else:
                target = build_answer_only_target(canonical)
                fallback += 1
        out.append({**rec, "response_vi": target, "_stage": f"stage_b_{STAGE_B_TARGET_MODE}"})
        if stable_fraction(rec["_source_id"]) < ANSWER_ONLY_REPLAY_RATIO:
            out.append({**rec, "response_vi": build_answer_only_target(canonical), "_stage": "stage_b_answer_replay"})
            replay += 1
    print(f"[build:stage_b] mode={STAGE_B_TARGET_MODE} selected={len(selected)} total={len(out)} answer_only={answer_only} reason_used={reason_used} fallback={fallback} replay={replay}")
    return out

valid_overlap_audit = save_valid_overlap_audit(train_records, valid_records, VALID_OVERLAP_AUDIT_PATH)
train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid")

train_stage_a = build_stage_a_records(train_clean)
valid_stage_a = build_stage_a_records(valid_clean)
train_stage_b = build_stage_b_records(train_clean)
valid_stage_b = valid_stage_a  # validation loss stays answer-focused

print("\nExample targets:")
print("[stage_a]", train_stage_a[0]["response_vi"])
print("[stage_b]", train_stage_b[0]["response_vi"][:500])


In [ ]:
# ============================================================
# 4. SFT dataset: pre-tokenized, loss only on response tokens
# ============================================================
class SFTDataset(Dataset):
    """Pre-tokenize once instead of tokenizing inside __getitem__.

    This removes a major CPU bottleneck from the Trainer loop. Each item keeps
    a `length` field so Trainer can bucket by length when group_by_length=True.
    """
    def __init__(self, records, tokenizer, max_length: int, desc: str = "train"):
        self.examples = []
        self.tok = tokenizer
        self.max_length = max_length
        for rec in tqdm(records, desc=f"tokenize:{desc}", leave=False):
            prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"])
            response = rec["response_vi"]
            p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
            r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

            if len(p_ids) >= self.max_length - 1:
                p_ids = p_ids[-(self.max_length - 1):]
            budget = self.max_length - len(p_ids)
            if budget <= 0:
                r_ids = [SAFE_EOS_ID]
            elif len(r_ids) > budget:
                # Keep the answer tail.
                r_ids = r_ids[-budget:]

            ids = p_ids + r_ids
            labels = [-100] * len(p_ids) + r_ids
            ids = [min(t, SAFE_EOS_ID) for t in ids]
            labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]
            self.examples.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
                "length": len(ids),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

@dataclass
class PadCollator:
    pad_id: int
    pad_to_multiple_of: int = 8

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            maxlen = ((maxlen + m - 1) // m) * m

        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

_probe_a = SFTDataset(train_stage_a[:1], tokenizer, MAX_LENGTH_STAGE_A, desc="probe_a")[0]
_probe_b = SFTDataset(train_stage_b[:1], tokenizer, MAX_LENGTH_STAGE_B, desc="probe_b")[0]
print("stage_a len/loss_tokens:", len(_probe_a["input_ids"]), sum(x != -100 for x in _probe_a["labels"]))
print("stage_b len/loss_tokens:", len(_probe_b["input_ids"]), sum(x != -100 for x in _probe_b["labels"]))
print("stage_a tail:", decode_model_text(tokenizer, _probe_a["input_ids"][-20:]))
print("stage_b tail:", decode_model_text(tokenizer, _probe_b["input_ids"][-30:]))


In [ ]:
# ============================================================
# 5. PEFT LoRA SFT: Stage A + Stage B
# ============================================================
def build_training_args(output_dir: Path, epochs: float, lr: float):
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        # Eval is disabled during training; keep this for compatibility only.
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy="no",          # save once manually after training
        report_to="none",
        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        group_by_length=True,        # reduce padding waste
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "no"
    else:
        kwargs["evaluation_strategy"] = "no"
    if "optim" in sig.parameters and torch.cuda.is_available():
        # Fallback automatically if unsupported by local transformers/torch.
        kwargs["optim"] = "adamw_torch_fused"
    return TrainingArguments(**kwargs)

def build_lora_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID

    # Do NOT enable gradient checkpointing by default for LoRA.
    # It saves VRAM but slows training because activations are recomputed.
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

def train_lora_stage(stage_name: str, model, train_records_for_stage, valid_records_for_stage, output_dir: Path, max_length: int, epochs: float, lr: float):
    print("\n" + "=" * 90)
    print(f"[train:{stage_name}] train={len(train_records_for_stage)} valid={len(valid_records_for_stage)} max_length={max_length} epochs={epochs} lr={lr}")
    train_ds = SFTDataset(train_records_for_stage, tokenizer, max_length, desc=stage_name)
    collator = PadCollator(SAFE_EOS_ID)

    eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
    print(f"[train:{stage_name}] eff_batch={eff_batch} steps/epoch={math.ceil(len(train_ds)/eff_batch)}")

    trainer = Trainer(
        model=model,
        args=build_training_args(output_dir, epochs, lr),
        train_dataset=train_ds,
        data_collator=collator,
    )
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    model_hash = sha256_dir(output_dir)
    (output_dir / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    print(f"[train:{stage_name}] wall={dt/60:.2f} min saved={output_dir} sha256={model_hash}")
    del trainer
    torch.cuda.empty_cache()
    return model, dt

model = build_lora_model()

model, stage_a_train_dt = train_lora_stage(
    "stage_a_answer_only_lora",
    model,
    train_stage_a,
    valid_stage_a,
    STAGE_A_OUTPUT_DIR,
    MAX_LENGTH_STAGE_A,
    STAGE_A_EPOCHS,
    STAGE_A_LR,
)

if RUN_STAGE_B and STAGE_B_EPOCHS > 0:
    model, stage_b_train_dt = train_lora_stage(
        "stage_b_local_reason_lora",
        model,
        train_stage_b,
        valid_stage_b,
        SFT_OUTPUT_DIR,
        MAX_LENGTH_STAGE_B,
        STAGE_B_EPOCHS,
        STAGE_B_LR,
    )
else:
    print("[train:stage_b] skipped for V6-simple; using Stage A adapter as SFT/final base.")
    SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(SFT_OUTPUT_DIR)
    tokenizer.save_pretrained(SFT_OUTPUT_DIR)
    model_hash = sha256_dir(SFT_OUTPUT_DIR)
    (SFT_OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    stage_b_train_dt = 0.0

print(f"[train:sft] total wall={(stage_a_train_dt + stage_b_train_dt)/60:.2f} min")


In [ ]:
# ============================================================
# 6. Custom GRPO-lite reward tuning
# ============================================================
def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())

def has_anchor(text: str) -> bool:
    return extract_answer(text) is not None

def has_reason_equation(text: str) -> bool:
    before = ANSWER_TAIL_RE.split(text or "")[0]
    return bool(re.search(r"\d", before) and re.search(r"[+\-*/=×÷^]|\\frac|\\sqrt", before))

def reward_completion(completion: str, gold_num: float | None, gen_token_count: int) -> dict:
    completion = clean_decoded_artifacts(completion)
    pred_answer = extract_answer(completion)
    pred_num = parse_number(pred_answer)
    error_value = rel_error(pred_num, gold_num)
    answer_score = score_one(error_value, pred_answer is not None)
    reward_answer = answer_score / 10.0
    reward_anchor = 0.15 if pred_answer is not None else -0.20
    reward_len = 0.05 if gen_token_count <= 64 else (-0.05 if gen_token_count > 96 else 0.0)
    reward_reason = 0.05 if has_reason_equation(completion) else 0.0
    reward = max(0.0, min(1.2, reward_answer + reward_anchor + reward_len + reward_reason))
    return {
        "reward": reward,
        "reward_answer": reward_answer,
        "reward_anchor": reward_anchor,
        "reward_len": reward_len,
        "reward_reason": reward_reason,
        "answer_score": answer_score,
        "extractable": pred_answer is not None,
        "pred_answer": pred_answer,
        "pred_num": pred_num,
        "rel_error": error_value,
    }

def next_cyclic_batch(records: list[dict], cursor: int, batch_size: int):
    batch = []
    for j in range(batch_size):
        batch.append(records[(cursor + j) % len(records)])
    return batch, (cursor + batch_size) % len(records)

def generate_rl_group(model, batch_records: list[dict]):
    prompts = []
    for rec in batch_records:
        prompts.extend([build_prompt(rec)] * RL_GROUP_SIZE)
    model.eval()
    old_cache = getattr(model.config, "use_cache", False)
    model.config.use_cache = True
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=N_POSITIONS - RL_MAX_NEW_TOKENS,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        seqs = model.generate(
            **enc,
            max_new_tokens=RL_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=RL_TEMPERATURE,
            top_p=RL_TOP_P,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
        )
    model.config.use_cache = old_cache
    model.train()
    prompt_width = enc["input_ids"].shape[1]
    gen_ids = seqs[:, prompt_width:]
    texts = [decode_model_text(tokenizer, row) for row in gen_ids]
    gen_counts = (gen_ids != SAFE_EOS_ID).sum(dim=1).detach().cpu().tolist()
    return enc, seqs, gen_ids, texts, gen_counts, prompt_width

def completion_logprob_mean(model, seqs: torch.Tensor, input_attention: torch.Tensor, prompt_width: int):
    full_attention = torch.ones_like(seqs)
    full_attention[:, :prompt_width] = input_attention
    out = model(input_ids=seqs, attention_mask=full_attention)
    logits = out.logits[:, :-1, :]
    targets = seqs[:, 1:]
    token_logp = F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    gen_ids = seqs[:, prompt_width:]
    gen_mask = (gen_ids != SAFE_EOS_ID).float()
    target_mask = torch.zeros_like(targets, dtype=torch.float32)
    target_mask[:, prompt_width - 1:] = gen_mask
    return (token_logp * target_mask).sum(dim=1) / target_mask.sum(dim=1).clamp(min=1.0)

def sft_replay_loss(model, records: list[dict]):
    replay = []
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is not None:
            replay.append({**rec, "response_vi": build_answer_only_target(canonical)})
    ds = SFTDataset(replay, tokenizer, MAX_LENGTH_STAGE_A)
    collator = PadCollator(SAFE_EOS_ID)
    batch = collator([ds[i] for i in range(len(ds))])
    batch = {k: v.to(model.device) for k, v in batch.items()}
    return model(**batch).loss

def run_grpo_lite(model):
    if not RL_ENABLED or RL_MAX_STEPS <= 0:
        print("[rl] disabled; saving SFT model as final.")
        FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(FINAL_OUTPUT_DIR)
        tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
        model_hash = sha256_dir(FINAL_OUTPUT_DIR)
        (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
        summary = {
            "enabled": False,
            "reason": "RL_ENABLED is False or RL_MAX_STEPS <= 0",
            "final_dir": str(FINAL_OUTPUT_DIR),
            "sha256": model_hash,
        }
        RL_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        return summary

    rl_records = build_balanced_subset(train_clean, RL_MAX_PROMPTS, SEED + 17)
    rng = random.Random(SEED + 19)
    rng.shuffle(rl_records)
    print(f"[rl] prompts={len(rl_records)} batch={RL_PROMPT_BATCH_SIZE} group={RL_GROUP_SIZE} steps={RL_MAX_STEPS}")

    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=RL_LR)
    cursor = 0
    log_rows = []
    t0 = time.time()
    RL_LOG_PATH.write_text("", encoding="utf-8")
    model.train()

    for step in tqdm(range(1, RL_MAX_STEPS + 1), desc="grpo-lite"):
        batch_records, cursor = next_cyclic_batch(rl_records, cursor, RL_PROMPT_BATCH_SIZE)
        enc, seqs, gen_ids, texts, gen_counts, prompt_width = generate_rl_group(model, batch_records)

        reward_infos = []
        for i, text in enumerate(texts):
            rec = batch_records[i // RL_GROUP_SIZE]
            reward_infos.append(reward_completion(text, rec.get("_gold_num"), gen_counts[i]))

        rewards = torch.tensor([x["reward"] for x in reward_infos], dtype=torch.float32, device=model.device)
        reward_group = rewards.view(len(batch_records), RL_GROUP_SIZE)
        advantages = (reward_group - reward_group.mean(dim=1, keepdim=True)).reshape(-1).detach()

        logprob_mean = completion_logprob_mean(model, seqs, enc["attention_mask"], prompt_width)
        rl_loss = -(advantages * logprob_mean).mean()
        replay_loss = sft_replay_loss(model, batch_records)
        total_loss = rl_loss + RL_SFT_REPLAY_COEF * replay_loss

        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), RL_GRAD_CLIP)
        optimizer.step()

        reward_mean = float(rewards.mean().detach().cpu())
        answer_reward_mean = sum(x["reward_answer"] for x in reward_infos) / len(reward_infos)
        extractable_rate = sum(x["extractable"] for x in reward_infos) / len(reward_infos)
        row = {
            "step": step,
            "loss": float(total_loss.detach().cpu()),
            "rl_loss": float(rl_loss.detach().cpu()),
            "sft_replay_loss": float(replay_loss.detach().cpu()),
            "reward_mean": reward_mean,
            "answer_reward_mean": answer_reward_mean,
            "extractable_rate": extractable_rate,
            "adv_abs_mean": float(advantages.abs().mean().detach().cpu()),
            "elapsed_min": (time.time() - t0) / 60,
        }
        log_rows.append(row)
        with RL_LOG_PATH.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
        if step == 1 or step % 25 == 0:
            print("[rl]", json.dumps(row, ensure_ascii=False))

    FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(FINAL_OUTPUT_DIR)
    tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
    model_hash = sha256_dir(FINAL_OUTPUT_DIR)
    (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")

    tail = log_rows[-min(50, len(log_rows)):]
    summary = {
        "enabled": True,
        "steps": len(log_rows),
        "rl_max_prompts": RL_MAX_PROMPTS,
        "group_size": RL_GROUP_SIZE,
        "prompt_batch_size": RL_PROMPT_BATCH_SIZE,
        "reward_mean_last50": sum(x["reward_mean"] for x in tail) / len(tail) if tail else None,
        "answer_reward_mean_last50": sum(x["answer_reward_mean"] for x in tail) / len(tail) if tail else None,
        "extractable_rate_last50": sum(x["extractable_rate"] for x in tail) / len(tail) if tail else None,
        "wall_min": (time.time() - t0) / 60,
        "final_dir": str(FINAL_OUTPUT_DIR),
        "sha256": model_hash,
    }
    RL_SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[rl-summary]", json.dumps(summary, ensure_ascii=False, indent=2))
    return summary

rl_summary = run_grpo_lite(model)

del model
torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 7. Generation + reports
# ============================================================
def has_peft_adapter(path_like) -> bool:
    p = Path(path_like)
    return p.exists() and (p / "adapter_config.json").exists()

def load_model_for_generation(adapter_dir: Path):
    dtype = torch.float16 if (INFER_FP16 and torch.cuda.is_available()) else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, local_files_only=True)
    base.config.pad_token_id = SAFE_EOS_ID
    base.config.eos_token_id = SAFE_EOS_ID
    if has_peft_adapter(adapter_dir):
        print(f"[infer] base + adapter: {adapter_dir}")
        gen_model = PeftModel.from_pretrained(base, str(adapter_dir), local_files_only=True)
        try:
            gen_model = gen_model.merge_and_unload()
            print("[infer] merged LoRA adapter")
        except Exception as exc:
            print("[infer] merge failed; using PEFT wrapper:", repr(exc))
    else:
        print(f"[infer] no adapter at {adapter_dir}; using base model")
        gen_model = base
    device = "cuda" if torch.cuda.is_available() else "cpu"
    gen_model.to(device)
    gen_model.eval()
    return gen_model

class StopOnAnswerLine(StoppingCriteria):
    """V3-style stopper for a single decoded row."""
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID, patience_tokens: int = 16):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 4:
            return False
        text = decode_model_text(self.tok, gen_tail)
        m = self.re_answer.search(text)
        if not m:
            return False
        if self._matched_at is None:
            self._matched_at = gen_tail.numel()
        if "\n" in text[m.end():]:
            return True
        if gen_tail.numel() - self._matched_at >= self.patience:
            return True
        return False

@torch.inference_mode()
def generate_outputs(adapter_dir: Path, records: list[dict], output_path: Path, *, max_new_tokens: int = MAX_NEW_TOKENS, num_beams: int = NUM_BEAMS):
    gen_tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    gen_tok.pad_token_id = SAFE_EOS_ID
    gen_tok.eos_token_id = SAFE_EOS_ID
    gen_tok.padding_side = "left"
    gen_tok.truncation_side = "left"
    gen_model = load_model_for_generation(adapter_dir)
    device = next(gen_model.parameters()).device
    outputs = []
    n_pos = int(getattr(gen_model.config, "n_positions", getattr(gen_model.config, "max_position_embeddings", 1024)))
    vocab_n = gen_model.get_input_embeddings().num_embeddings
    for idx, rec in enumerate(tqdm(records, desc=f"generate:{Path(adapter_dir).name}")):
        prompt = build_prompt(rec)
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = gen_tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        full_ids = [min(t, vocab_n - 1) for t in full_ids]
        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]
        eff_new = max(8, min(max_new_tokens, n_pos - prompt_len))
        gen_kwargs = dict(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            stopping_criteria=StoppingCriteriaList([StopOnAnswerLine(gen_tok, prompt_len=prompt_len)]),
        )
        if num_beams and num_beams > 1:
            gen_kwargs.update(dict(num_beams=num_beams, do_sample=False, early_stopping=True, length_penalty=LENGTH_PENALTY))
        else:
            gen_kwargs.update(dict(num_beams=1, do_sample=False))
        seqs = gen_model.generate(**gen_kwargs)
        text = decode_model_text(gen_tok, seqs[0, prompt_len:])
        if SANITIZE_TO_ANSWER_ONLY:
            text = sanitize_model_output(text)
        outputs.append({
            "id": idx,
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "model_output": text.strip(),
        })
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[infer] wrote {len(outputs)} rows -> {output_path}")
    del gen_model
    torch.cuda.empty_cache()
    return outputs

if RUN_MODE == "phase1":
    if RUN_STAGE_A_GENERATION_EVAL:
        _ = generate_outputs(STAGE_A_OUTPUT_DIR, valid_records, STAGE_A_VALID_OUTPUT_PATH, max_new_tokens=32, num_beams=NUM_BEAMS)
        rep = save_eval_report(STAGE_A_VALID_OUTPUT_PATH, valid_records, STAGE_A_VALID_REPORT_PATH)
        print("[stage_a]", rep["summary"])

    if RUN_SFT_GENERATION_EVAL:
        _ = generate_outputs(SFT_OUTPUT_DIR, valid_records, SFT_VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
        rep = save_eval_report(SFT_VALID_OUTPUT_PATH, valid_records, SFT_VALID_REPORT_PATH)
        print("[sft]", rep["summary"])

    _ = generate_outputs(FINAL_OUTPUT_DIR, valid_records, VALID_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    rep = save_eval_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
    print("[final]", rep["summary"])
    print("Score /10:", rep["summary"]["score_10"])

elif RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"RUN_MODE='phase2' but missing {TEST_FILE}")
    test_records = load_records(TEST_FILE)
    _ = generate_outputs(FINAL_OUTPUT_DIR, test_records, TEST_OUTPUT_PATH, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    print("[phase2] wrote", TEST_OUTPUT_PATH)
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE}")


In [ ]:
# ============================================================
# 8. Output manifest
# ============================================================
manifest = {
    "run_mode": RUN_MODE,
    "use_kd": USE_KD,
    "stage_a_output_dir": str(STAGE_A_OUTPUT_DIR),
    "sft_output_dir": str(SFT_OUTPUT_DIR),
    "final_output_dir": str(FINAL_OUTPUT_DIR),
    "valid_output_stage_a": str(STAGE_A_VALID_OUTPUT_PATH),
    "valid_report_stage_a": str(STAGE_A_VALID_REPORT_PATH),
    "valid_output_sft": str(SFT_VALID_OUTPUT_PATH),
    "valid_report_sft": str(SFT_VALID_REPORT_PATH),
    "valid_output": str(VALID_OUTPUT_PATH),
    "valid_report": str(VALID_REPORT_PATH),
    "valid_overlap_audit": str(VALID_OVERLAP_AUDIT_PATH),
    "rl_training_log": str(RL_LOG_PATH),
    "rl_reward_summary": str(RL_SUMMARY_PATH),
    "test_predictions": str(TEST_OUTPUT_PATH),
}
manifest_path = WORKING_DIR / "v6_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
